# soft_prompt_iii_train — variant (iii), prefix-tuning, full run
Trains learnable key/value pairs injected into **every** MLLM attention layer via `past_key_values`. This is a layerwise K/V prefix-tuning implementation closely aligned with the mechanism Petrov's Section 4 analyses — Petrov derives K and V from a shared prefix vector through frozen projections, whereas this learns them directly, so the two coincide only under a rank condition the released weights do not let us check — a prefix can bias each attention block's output but cannot change the relative attention over the content — and the only variant whose parameter count reaches LoRA rank 4.

Dimensions measured from the released config by `soft_prompt_iii_probe`: hidden 2048, 36 layers, 2 KV heads, head_dim 128, so K/V are 256 wide per layer under grouped-query attention, giving 18,432 parameters per prefix token. **p = 139 matches LoRA rank 4 at 100.3%** (2,562,048 against 2,553,600), and prefix plus instruction sits well inside the 888-token training cap — measured against the training pool in section 4.

Also measured: 12.45 s/step, the cheapest of the three variants, and lr 1e-3 giving +3.8% prefix drift over 15 post-warmup steps against +31.6% at 3e-3. The gradient column is uninformative here — uniformly ~1e-4, roughly 40x below variant (ii), consistent with a prefix sitting behind 36 frozen layers — so the rate was chosen on drift.

The prefix is attended to but produces **no output rows**: `text_feats` keeps one row per input token. The extended mask is for the encoder call only; `text_mask` must stay its original length for the diffusion transformer, or its rotary embedding raises a size mismatch.

Validation loss is logged every 250 steps on the same 98-row slice, timestep draws and seed the LoRA arm used. Not as an early-stopping signal — the LoRA arm established it runs opposite to the task metric — but as a replication test of that finding: if the same inversion appears here, that is evidence the mismatch is not unique to LoRA, and is consistent with a pipeline- or evaluation-level effect without identifying which.

3000 steps, prefix written every 500, four scored later at 500 / 1000 / 2000 / 3000.

## 1. Repo, deps

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
    # requirements.txt omits torchdiffeq (omnigen2/transport/integrators.py) and peft (train.py)
    subprocess.run([sys.executable,"-m","pip","install","-q","torchdiffeq"], check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

# peft's is_torchao_available() raises rather than returning False when torchao predates 0.16;
# Colab ships 0.10. torchao is unused by this bf16 path. Guarded, so a no-op once gone.
_ao = subprocess.run([sys.executable,"-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","torchao"], check=True)
    print("torchao removed")
sys.path.insert(0, REPO); os.chdir(REPO)
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

installed.
torchao removed
NVIDIA A100-SXM4-40GB, 40960 MiB


## 2. Drive, images, splits

In [ ]:
from google.colab import drive, userdata
import json
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                   shell=True, check=True)
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val"]:
    src = f"{DRIVE}/{s}.jsonl"
    if not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"], check=True)
subprocess.run(["cp", f"{DRIVE}/screen_exclude_from_training.txt", "/content/"], check=True)
print("splits restored")

Mounted at /content/drive
restored 37063 / 37063
splits restored


## 3. Prefix dimensions, initialisation, run identity
Reads `num_hidden_layers`, `num_key_value_heads` and `head_dim` from the released MLLM config and derives the per-token parameter count as `2 (K and V) x num_key_value_heads x head_dim x num_hidden_layers`, then sets `P` to the length that matches LoRA rank 4.

Initialised as small Gaussian noise at 0.02, the standard prefix-tuning initialisation, and the one the probe validated.

In [ ]:
import torch, re
from transformers import AutoConfig

LR   = 1e-3        # from soft_prompt_iii_probe, chosen on prefix drift
STEPS = 3000
CKPT_EVERY = 500
SEED = 2233
MODEL = "OmniGen2/OmniGen2"
torch.manual_seed(SEED)

cfg_mllm = AutoConfig.from_pretrained(MODEL, subfolder="mllm", trust_remote_code=True)
tc  = getattr(cfg_mllm, "text_config", cfg_mllm)
H   = tc.hidden_size
L   = tc.num_hidden_layers
NKV = getattr(tc, "num_key_value_heads", tc.num_attention_heads)
HD  = getattr(tc, "head_dim", H // tc.num_attention_heads)
PER_TOKEN = 2 * NKV * HD * L
LORA_R4 = 38 * 16800 * 4
P = round(LORA_R4 / PER_TOKEN)
RUN = f"sp_iii_p{P}"

print(f"hidden {H} | layers {L} | kv heads {NKV} | head_dim {HD}")
print(f"per prefix token = 2 x {NKV} x {HD} x {L} = {PER_TOKEN:,} params")
# prefix + instruction length is checked against the measured training pool in section 4

init = torch.randn(L, 2, NKV, P, HD) * 0.02
EXP = f"{DRIVE}/soft_prompt/experiments/{RUN}"
os.makedirs(EXP, exist_ok=True); os.makedirs("/content/sp", exist_ok=True)
torch.save(init, "/content/sp/init.pt")
json.dump({"P": P, "layers": L, "n_kv": NKV, "head_dim": HD, "per_token": PER_TOKEN,
           "init_path": "/content/sp/init.pt", "seed": SEED, "save_dir": EXP, "every": CKPT_EVERY},
          open("/content/soft_prompt.json", "w"))

print(f"\nrun {RUN}  prefix {tuple(init.shape)}  trainable={init.numel():,} "
      f"= {100*init.numel()/LORA_R4:.0f}% of LoRA r4")
print(f"lr {LR}  steps {STEPS}  checkpoints every {CKPT_EVERY} -> {EXP}")
print(f"init norm {init.norm():.3f}  (the probe measured 32.033)")

config.json: 0.00B [00:00, ?B/s]

hidden 2048 | layers 36 | kv heads 2 | head_dim 128
per prefix token = 2 x 2 x 128 x 36 = 18,432 params

run sp_iii_p139  prefix (36, 2, 2, 139, 128)  trainable=2,562,048 = 100% of LoRA r4
lr 0.001  steps 3000  checkpoints every 500 -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/soft_prompt/experiments/sp_iii_p139
init norm 32.032  (the probe measured 32.033)


## 4. Training data
The full domain training split with the 62 screen targets excluded, in OmniGen2's format. Identical row set to the LoRA arm.

The length diagnostic below counts the raw instruction rather than the chat-templated string the collator truncates, so its 263-token maximum and 402-position figure are both 25 tokens short. On the templated basis measured in `inspect_template_surface` the maximum is 288 and the prefix plus instruction reaches 427, still inside 888 for 100% of training rows, so the assertion and the conclusion hold on either basis. The output is retained as executed.

In [ ]:
rows = [json.loads(l) for l in open("/content/train.jsonl")]
excl = {l.strip() for l in open("/content/screen_exclude_from_training.txt") if l.strip()}
rows = [r for r in rows if r["output_image"] not in excl]
assert len(rows) == 39591, f"expected 39591 training rows, got {len(rows)}"
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained(MODEL, subfolder="processor", trust_remote_code=True)
_len = [len(x) for x in _tok([r["instruction"] for r in rows], add_special_tokens=False).input_ids]
MAX_INSTR = max(_len)
print(f"longest training instruction {MAX_INSTR} tokens; prefix {P} + {MAX_INSTR} = {P + MAX_INSTR} attended positions inside the MLLM")
print(f"{100*sum(1 for n in _len if P + n <= 888)/len(_len):.2f}% of training rows keep prefix+instruction inside the 888 seen in training")
assert P + MAX_INSTR <= 888, f"prefix {P} plus longest training instruction {MAX_INSTR} exceeds the 888-token cap"
print(f"train rows {len(rows)}  (39,893 minus the 302 screen rows)")

os.makedirs("/content/data", exist_ok=True)
with open("/content/data/train_sp.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps({
            "task_type": "ic",
            "instruction": r["instruction"],
            "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
            "output_image": os.path.join(IMGS, r["output_image"]),
        }) + "\n")

os.makedirs(f"{REPO}/data_configs/train/sp", exist_ok=True)
with open(f"{REPO}/data_configs/train/sp/sp.yml", "w") as f:
    f.write("data:\n  -\n    path: /content/data/train_sp.jsonl\n    type: 'ic'\n    ratio: !!float 1\n")
print("data config written")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

processor/tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

longest training instruction 263 tokens; prefix 139 + 263 = 402 attended positions inside the MLLM
100.00% of training rows keep prefix+instruction inside the 888 seen in training
train rows 39591  (39,893 minus the 302 screen rows)
data config written


## 5. Patch train.py — learnable K/V at every MLLM layer
**A** freezes everything and hands the optimizer the prefix.

**B** replaces the entire `torch.no_grad()` block. The prefix is passed as `past_key_values`, which is how transformers prepends cached keys and values to every layer's attention — prefix-tuning's mechanism. `_mask_ext` covers the prefix positions for the encoder call only; `text_mask` stays its original length for the diffusion transformer. An assertion enforces that the prefix adds no output rows, which is the failure the probe caught.

In [ ]:
p = f"{REPO}/train.py"
src = open(p, encoding="utf-8").read()

A = "    trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))\n"
A_NEW = "\n".join([
    "    # prefix-tuning variant (iii): learnable K/V for every MLLM layer, shape [L,2,n_kv,P,head_dim]",
    "    import json as _json",
    "    _spc = _json.load(open('/content/soft_prompt.json'))",
    "    model.requires_grad_(False)",
    "    text_encoder.requires_grad_(False)   # no_grad is removed below, so without this the",
    "    vae.requires_grad_(False)            # backward stores .grad for all 3B frozen MLLM params",
    "    PREFIX = torch.nn.Parameter(",
    "        torch.load(_spc['init_path'], map_location='cpu').to(accelerator.device, dtype=torch.float32))",
    "    PREFIX.requires_grad_(True)",
    "    globals()['PREFIX'] = PREFIX",
    "    globals()['SP_CFG'] = _spc",
    "    trainable_params = [PREFIX]",
    "    logger.info('[soft_prompt] variant(iii) shape=%s trainable=%d' % (",
    "        tuple(PREFIX.shape), PREFIX.numel()))",
]) + "\n"

B = "\n".join([
    "                with torch.no_grad():",
    "                    text_feats = text_encoder(",
    "                        input_ids=text_input_ids,",
    "                        attention_mask=text_mask,",
    "                        output_hidden_states=False,",
    "                    ).last_hidden_state",
]) + "\n"
B_NEW = "\n".join([
    "                # variant (iii): learnable K/V prepended at every layer via past_key_values.",
    "                # no_grad removed so the loss reaches the prefix; encoder weights stay frozen.",
    "                from transformers.cache_utils import DynamicCache",
    "                _B = text_input_ids.shape[0]",
    "                _pf = PREFIX.to(dtype=text_encoder.dtype)",
    "                _legacy = tuple(",
    "                    (_pf[_l, 0].unsqueeze(0).expand(_B, -1, -1, -1),",
    "                     _pf[_l, 1].unsqueeze(0).expand(_B, -1, -1, -1))",
    "                    for _l in range(_pf.shape[0]))",
    "                _cache = DynamicCache.from_legacy_cache(_legacy)",
    "                # the prefix is attended to but produces NO output rows: text_feats keeps one row",
    "                # per input token. So the extended mask is for the encoder call only; text_mask",
    "                # must stay its original length for the diffusion transformer at line 546.",
    "                _mask_ext = torch.cat([",
    "                    torch.ones(_B, _pf.shape[3], dtype=text_mask.dtype,",
    "                               device=text_mask.device), text_mask], dim=1)",
    "                text_feats = text_encoder(",
    "                    input_ids=text_input_ids,",
    "                    attention_mask=_mask_ext,",
    "                    past_key_values=_cache,",
    "                    use_cache=False,",
    "                    output_hidden_states=False,",
    "                ).last_hidden_state",
    "                assert text_feats.shape[1] == text_input_ids.shape[1], (",
    "                    'prefix must not add output rows: %d vs %d' % (text_feats.shape[1], text_input_ids.shape[1]))",
    "                if not text_feats.requires_grad:",
    "                    raise RuntimeError('prefix is not in the graph - variant (iii) patch')",
]) + "\n"

if "variant(iii)" in src:
    print("already patched")
else:
    assert src.count(A) == 1, f"anchor A found {src.count(A)} times"
    assert src.count(B) == 1, f"anchor B found {src.count(B)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A, A_NEW).replace(B, B_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("patched and parses")

patched and parses


## 5a. Loss normalisation
gradient_state.num_steps is 1 on the reference 8-GPU config and 8 here (single GPU, accum 8),
so the shipped normalisation runs at 8x the reference gradient scale. Corrected to match the
LoRA arm, which applies the same replacement. Derivation in lora_train_r{4,8,16} section 5.

In [ ]:
FIX_ACCUM_NORM = True

p = f"{REPO}/train.py"
A4     = ("                loss = (loss * accelerator.gradient_state.num_steps"
          " * accelerator.num_processes) / num_tokens_in_batch\n")
A4_NEW = "                loss = (loss * accelerator.num_processes) / num_tokens_in_batch\n"

src = open(p, encoding="utf-8").read()
if A4_NEW in src:
    print("accum-norm already corrected")
elif not FIX_ACCUM_NORM:
    print("FIX_ACCUM_NORM=False - shipped normalisation retained (8x reference gradient scale)")
else:
    assert src.count(A4) == 1, f"anchor A4 found {src.count(A4)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A4, A4_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("accum-norm corrected and train.py parses")

now = open(p, encoding="utf-8").read()
print(f"  accum-norm fix : {A4_NEW in now}  (FIX_ACCUM_NORM={FIX_ACCUM_NORM})")
print(f"  shipped line   : {A4 in now}")

accum-norm corrected and train.py parses
  accum-norm fix : True  (FIX_ACCUM_NORM=True)
  shipped line   : False


## 6. Checkpointing and monitor
`accelerator.save_state` captures the model, and the prefix is deliberately not part of it, so **S** writes the prefix itself at every checkpoint interval. Each file is a few MB.

**M** prints the gradient norm and the prefix norm every 50 steps. Over a 10-hour run this is the live answer to whether the arm is learning: a gradient that dies or a prefix norm that stops moving means the result would be undertrained by construction rather than by expressiveness. The probe's readings at lr 1e-3 were gradient 1.530e-04 and prefix norm rising 32.033 -> 33.246 over 15 steps. The gradient is roughly 40x below variant (ii)'s, consistent with a prefix sitting behind 36 frozen layers.

In [ ]:
src = open(p, encoding="utf-8").read()

S = "                if global_step % args.logger.checkpointing_steps == 0:\n"
S_NEW = "\n".join([
    "                if global_step % SP_CFG['every'] == 0 and accelerator.is_main_process:",
    "                    import os as _os",
    "                    _d = SP_CFG['save_dir']",
    "                    _os.makedirs(_d, exist_ok=True)",
    "                    torch.save({'prefix': PREFIX.detach().cpu(),",
    "                                'step': global_step, 'P': PREFIX.shape[3],",
    "                                'shape': tuple(PREFIX.shape)},",
    "                               _os.path.join(_d, 'prompt-%d.pt' % global_step))",
    "                    logger.info('[sp] saved prompt-%d.pt' % global_step)",
]) + "\n"

M = "                    accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)\n"
M_NEW = "\n".join([
    # clip_grad_norm_ returns the PRE-clip total norm; reading .grad after the call
    # returns the clipped value, which saturates at max_grad_norm and is not a measurement
    "                    _gnorm = accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)",
    "                    if accelerator.is_main_process and global_step % 50 == 0:",
    "                        print('[sp] step %d grad_norm %.3e prefix_norm %.4f' % (",
    "                            global_step, float(_gnorm),",
    "                            PREFIX.detach().float().norm().item()), flush=True)",
]) + "\n"

if "[sp] saved" in src:
    print("already patched")
else:
    assert src.count(S) == 1, f"anchor S found {src.count(S)} times"
    assert src.count(M) == 1, f"anchor M found {src.count(M)} times"
    open(p, "w", encoding="utf-8").write(src.replace(S, S_NEW + S).replace(M, M_NEW))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("checkpoint and monitor patched")

checkpoint and monitor patched


## 7. Validation loss module
The evaluator the LoRA arm used, plus an `encode_text` hook. Without it the val path would call the frozen text encoder directly and measure the model with no prefix attached — a flat curve that looks like a result.

In [ ]:
VAL_MODULE = '"""Held-out validation loss for OmniGen2 training.\n\nEvaluates a deterministic held-out proxy for the training objective, using the VAE posterior mode rather than a posterior sample so successive evaluations isolate weight changes, on a held-out split with gradients disabled.\nTimesteps and noise are drawn through the repository\'s own transport.sample under\na fixed seed, so the timestep distribution and resolution-dependent shift match\ntraining exactly while successive evaluations differ only by the model weights.\n\nGlobal RNG state is saved and restored around every evaluation so that the\ntraining stream is unaffected.\n\nNot part of the upstream OmniGen2 repository.\n"""\n\nimport json\nimport os\nimport time\n\nimport torch\n\n\nclass ValLossEvaluator:\n    """Per-token velocity MSE on a held-out split at fixed timesteps and noise."""\n\n    def __init__(self, dataset, collator, transport, out_path, batch_size=2,\n                 every=250, n_repeats=4, seed=1234, encode_text=None):\n        self.dataset = dataset\n        self.collator = collator\n        self.transport = transport\n        self.out_path = out_path\n        self.batch_size = batch_size\n        self.every = every\n        self.n_repeats = n_repeats\n        self.seed = seed\n        self.encode_text = encode_text\n        d = os.path.dirname(out_path)\n        if d:\n            os.makedirs(d, exist_ok=True)\n\n    @torch.no_grad()\n    def run(self, model, vae, text_encoder, freqs_cis, weight_dtype, global_step, device):\n        t_start = time.time()\n        was_training = model.training\n        model.eval()\n\n        cpu_rng = torch.get_rng_state()\n        cuda_rng = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None\n\n        sse = 0.0\n        tokens = 0.0\n        by_t = []\n\n        try:\n            for start in range(0, len(self.dataset), self.batch_size):\n                idx = list(range(start, min(start + self.batch_size, len(self.dataset))))\n                batch = self.collator([self.dataset[i] for i in idx])\n\n                text_ids = batch["text_ids"].to(device)\n                text_mask = batch["text_mask"].to(device)\n                if self.encode_text is None:\n                    text_feats = text_encoder(\n                        input_ids=text_ids,\n                        attention_mask=text_mask,\n                        output_hidden_states=False,\n                    ).last_hidden_state\n                else:\n                    # soft-prompt arms: mirror the training-loop injection so val loss is\n                    # measured on the same conditioning the optimiser is shaping\n                    text_feats, text_mask = self.encode_text(text_encoder, text_ids, text_mask)\n\n                def encode(img):\n                    z = vae.encode(img.to(device=device, dtype=vae.dtype)).latent_dist.mode()\n                    if vae.config.shift_factor is not None:\n                        z = z - vae.config.shift_factor\n                    if vae.config.scaling_factor is not None:\n                        z = z * vae.config.scaling_factor\n                    return z.to(dtype=weight_dtype)\n\n                ref_latents = []\n                for imgs in batch["input_images"]:\n                    if imgs is not None and len(imgs) > 0:\n                        ref_latents.append([encode(im).squeeze(0) for im in imgs])\n                    else:\n                        ref_latents.append(None)\n\n                x1 = [encode(im).squeeze(0) for im in batch["output_image"]]\n\n                model_kwargs = dict(\n                    text_hidden_states=text_feats,\n                    text_attention_mask=text_mask,\n                    ref_image_hidden_states=ref_latents,\n                    freqs_cis=freqs_cis,\n                )\n\n                for k in range(self.n_repeats):\n                    torch.manual_seed((self.seed * 100003 + start * 101 + k) % (2 ** 31 - 1))\n                    t, x0, _ = self.transport.sample(x1, 0, 1)\n                    t, xt, ut = self.transport.path_sampler.plan(t, x0, x1)\n                    pred = model(xt, t, **dict(model_kwargs))\n                    for j in range(len(x1)):\n                        d = (pred[j].float() - ut[j].float()) ** 2\n                        sse += float(d.sum().item())\n                        tokens += float(d.numel())\n                        by_t.append((float(t[j].item()), float(d.mean().item())))\n        finally:\n            torch.set_rng_state(cpu_rng)\n            if cuda_rng is not None:\n                torch.cuda.set_rng_state_all(cuda_rng)\n            if was_training:\n                model.train()\n\n        val_loss = sse / max(tokens, 1.0)\n\n        bins = [0.0] * 4\n        counts = [0] * 4\n        for t_val, mse in by_t:\n            b = min(int(t_val * 4), 3)\n            bins[b] += mse\n            counts[b] += 1\n        binned = {f"t_{i/4:.2f}_{(i+1)/4:.2f}": (bins[i] / counts[i] if counts[i] else None)\n                  for i in range(4)}\n\n        with open(self.out_path, "a", encoding="utf-8") as f:\n            f.write(json.dumps({\n                "step": int(global_step),\n                "val_loss": val_loss,\n                "n_rows": len(self.dataset),\n                "n_repeats": self.n_repeats,\n                "n_samples": len(by_t),\n                "seconds": round(time.time() - t_start, 1),\n                "by_timestep_quartile": binned,\n            }) + "\\n")\n\n        return val_loss\n'

path = f"{REPO}/omnigen2/val_loss.py"
open(path, "w", encoding="utf-8").write(VAL_MODULE)
import ast; ast.parse(open(path, encoding='utf-8').read())
print(f"wrote {path} ({len(VAL_MODULE)} chars) and it parses")

wrote /content/OmniGen2/omnigen2/val_loss.py (5257 chars) and it parses


## 8. Validation loss patch
The 98-row val slice, written in the same form as the training pool, and the two anchors that install the evaluator. `_sp_encode` mirrors patch B exactly, so validation loss is measured on the conditioning the optimiser is actually shaping.

In [ ]:
# validation loss on the frozen 98-row val slice: same rows, timestep draws and seed
# as the LoRA arm, so the two curves are directly comparable
val_rows = [json.loads(l) for l in open("/content/val.jsonl", encoding="utf-8")]
assert len(val_rows) == 98, f"expected 98 val rows, got {len(val_rows)}"
assert not ({r["output_image"] for r in rows} & {r["output_image"] for r in val_rows}), \
    "val targets leaked into training"

with open("/content/data/val_sp.jsonl", "w", encoding="utf-8") as f:
    for r in val_rows:
        f.write(json.dumps({
            "task_type": "ic",
            "instruction": r["instruction"],
            "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
            "output_image": os.path.join(IMGS, r["output_image"]),
        }) + "\n")
with open(f"{REPO}/data_configs/train/sp/val_sp.yml", "w") as f:
    f.write("data:\n  -\n    path: /content/data/val_sp.jsonl\n    type: 'ic'\n    ratio: !!float 1\n")
print(f"val config written: {len(val_rows)} rows")

src = open(p, encoding="utf-8").read()
A1 = '    logger.info(f"{args.train.batch_size=} {args.train.gradient_accumulation_steps=} {accelerator.num_processes=} {args.train.global_batch_size=}")\n'
A2 = "                global_step += 1\n"

SETUP = '''    # held-out validation loss (not part of upstream OmniGen2)
    _val_eval = None
    if args.get("val_loss", None) is not None and accelerator.is_main_process:
        from omnigen2.val_loss import ValLossEvaluator
        _val_dataset = OmniGen2TrainDataset(
            args.val_loss.data_path,
            tokenizer=text_tokenizer,
            use_chat_template=args.data.use_chat_template,
            prompt_dropout_prob=0.0,
            ref_img_dropout_prob=0.0,
            max_input_pixels=OmegaConf.to_object(args.data.get("max_input_pixels", 1024 * 1024)),
            max_output_pixels=args.data.get("max_output_pixels", 1024 * 1024),
            max_side_length=args.data.get("max_side_length", 2048),
        )
        # variant (iii): learnable K/V as past_key_values at every MLLM layer
        def _sp_encode(enc, ids, mask):
            from transformers.cache_utils import DynamicCache
            _B = ids.shape[0]
            _pf = PREFIX.to(dtype=enc.dtype)
            _cache = DynamicCache.from_legacy_cache(tuple(
                (_pf[_l, 0].unsqueeze(0).expand(_B, -1, -1, -1),
                 _pf[_l, 1].unsqueeze(0).expand(_B, -1, -1, -1))
                for _l in range(_pf.shape[0])))
            _mask_ext = torch.cat([torch.ones(_B, _pf.shape[3], dtype=mask.dtype,
                                              device=mask.device), mask], dim=1)
            _f = enc(input_ids=ids, attention_mask=_mask_ext, past_key_values=_cache,
                     use_cache=False, output_hidden_states=False).last_hidden_state
            assert _f.shape[1] == ids.shape[1], "prefix must not add output rows"
            return _f, mask
        _val_eval = ValLossEvaluator(
            dataset=_val_dataset,
            collator=OmniGen2Collator(tokenizer=text_tokenizer,
                                      max_token_len=args.data.maximum_text_tokens),
            transport=transport,
            out_path=args.val_loss.out_path,
            batch_size=args.val_loss.get("batch_size", 2),
            every=args.val_loss.get("every", 250),
            n_repeats=args.val_loss.get("n_repeats", 4),
            seed=args.val_loss.get("seed", 1234),
            encode_text=_sp_encode,
        )
        logger.info(f"[val] {len(_val_dataset)} rows every {_val_eval.every} steps, "
                    f"{_val_eval.n_repeats} draws per row")

'''

CALL = '''                if _val_eval is not None and global_step % _val_eval.every == 0:
                    try:
                        _vl = _val_eval.run(model, vae, text_encoder, freqs_cis,
                                            weight_dtype, global_step, accelerator.device)
                        logs["val_loss"] = _vl
                        logger.info("[val] step %d val_loss %.6f" % (global_step, _vl))
                    except Exception as _e:
                        logger.warning("[val] step %d failed: %r" % (global_step, _e))
'''

if "_val_eval" in src:
    print("val hook already patched")
else:
    assert src.count(A1) == 1, f"anchor A1 found {src.count(A1)} times"
    assert src.count(A2) == 1, f"anchor A2 found {src.count(A2)} times"
    open(p, "w", encoding="utf-8").write(src.replace(A1, SETUP + A1).replace(A2, A2 + CALL))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("val hook patched")

val config written: 98 rows
val hook patched


## 9. Config
`ft_lora.yml` with `lora_ft` off, the three resolution keys at 512, and `gradient_accumulation_steps: 8` so `global_batch_size: 16` holds on one GPU — the same deviations the LoRA runs record. `warmup_t` is back to the shipped 500, unlike the probe, so the schedule shape matches the LoRA arm. `checkpointing_steps` is set beyond the run length and patch S inserts the prefix write above the shipped checkpoint block rather than replacing its condition, so `accelerator.save_state` stayed gated on `checkpointing_steps` and never fired: no accelerate state directories were written. The prefix tensors patch S writes are the checkpoints this arm evaluates.

In [ ]:
import yaml
cfg = yaml.safe_load(open(f"{REPO}/options/ft_lora.yml"))
cfg["train"]["lora_ft"] = False
cfg["train"]["batch_size"] = 2
cfg["train"]["gradient_accumulation_steps"] = 8
cfg["train"]["global_batch_size"] = 16
cfg["train"]["max_train_steps"] = STEPS
cfg["train"]["learning_rate"] = LR
cfg["train"]["gradient_checkpointing"] = True
cfg["name"] = RUN     # train.py:81 derives output_dir from conf.name; there is no train.output_dir
cfg["logger"]["log_with"] = ["tensorboard"]        # wandb would block on an interactive login
cfg["logger"]["checkpointing_steps"] = STEPS * 10  # gates the shipped save_state block out of
                                                   # a 3000-step run; patch S writes the prompt itself
cfg["data"]["max_output_pixels"] = 512*512
cfg["data"]["max_input_pixels"] = [512*512]*4
cfg["data"]["max_side_length"] = 512
cfg["data"]["data_path"] = "data_configs/train/sp/sp.yml"
cfg["val_loss"] = {                                # 98 rows, 4 fixed-seed draws, as the LoRA arm
    "data_path": "data_configs/train/sp/val_sp.yml",
    "out_path": f"{EXP}/val_loss.jsonl",
    "every": 250,
    "batch_size": 2,
    "n_repeats": 4,
    "seed": 1234,
}

OPT = f"/content/{RUN}.yml"
yaml.safe_dump(cfg, open(OPT, "w"), sort_keys=False)
print(f"lr {cfg['train']['learning_rate']} | steps {cfg['train']['max_train_steps']} | "
      f"warmup_t {cfg['train']['warmup_t']} | maximum_text_tokens {cfg['data']['maximum_text_tokens']}")
print(f"batch {cfg['train']['batch_size']} x accum {cfg['train']['gradient_accumulation_steps']} "
      f"= global {cfg['train']['global_batch_size']}")
print(f"val loss every {cfg['val_loss']['every']} steps -> {cfg['val_loss']['out_path']}")

lr 0.001 | steps 3000 | warmup_t 500 | maximum_text_tokens 888
batch 2 x accum 8 = global 16
val loss every 250 steps -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/soft_prompt/experiments/sp_iii_p139/val_loss.jsonl


## 10. Train
NOT resumable. Patch S inserts the prefix write above the shipped checkpoint block and leaves its condition alone, so `accelerator.save_state` stayed gated on `checkpointing_steps` and never fired. It would not have helped — the prefix is a bare Parameter outside `model`, so `save_state` does not capture it, and `resume_from_checkpoint: latest` would restore weights, optimizer moments and the step counter onto a prefix reset to init. This run went 0 to 3000 in a single session, logged steps advancing by 50 with no gap, so no resume occurred. Patch S is the mitigation: a run that dies at 2000 still leaves four prompts, three of them scoring points. Expect roughly 10.4 h from the probe's measured 12.45 s/step at P=139.

In [ ]:
# every patch must be present before a 10-hour run starts: a reverted or half-patched
# train.py would otherwise train silently wrong or crash after the model has loaded
_s = open(p, encoding='utf-8').read()
for _k in ('variant(iii)', '[sp] saved', '[sp] step', '_val_eval'):
    assert _k in _s, f'train.py is missing patch marker {_k!r} - re-run sections 5 to 8'
assert 'loss = (loss * accelerator.num_processes) / num_tokens_in_batch' in _s, \
    'train.py loss normalisation not corrected - re-run section 5a'
print('all patches present in train.py')
import time, re
t0 = time.time()
proc = subprocess.Popen(["accelerate","launch","--num_processes=1","--mixed_precision=bf16",
                         "train.py","--config",OPT],
                        cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env={**os.environ, "WANDB_DISABLED": "true"})
LOG = f"/content/{RUN}_console.log"
with open(LOG, "w") as fh:
    for line in proc.stdout:
        fh.write(line)
        if re.search(r"\[sp\]|\[soft_prompt\]|Error|Traceback|error", line):
            print(f"[{(time.time()-t0)/3600:5.2f}h] {line.rstrip()[:150]}", flush=True)
proc.wait()
print(f"\nexit {proc.returncode} after {(time.time()-t0)/3600:.2f}h")
subprocess.run(["cp", LOG, f"{EXP}/console.log"])

all patches present in train.py
[ 0.00h] 2026-08-01 18:01:10.646272: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due 
[ 0.03h] 08/01/2026 18:03:00 - INFO - __main__ - [RANK 0] [soft_prompt] variant(iii) shape=(36, 2, 2, 139, 128) trainable=2562048
[ 0.05h] [sp] step 0 grad_norm 5.952e-04 prefix_norm 32.0331
[ 0.22h] [sp] step 50 grad_norm 2.903e-04 prefix_norm 32.0601
[ 0.40h] [sp] step 100 grad_norm 1.657e-03 prefix_norm 32.3969
[ 0.58h] [sp] step 150 grad_norm 1.704e-04 prefix_norm 33.8871
[ 0.76h] [sp] step 200 grad_norm 1.741e-04 prefix_norm 37.2261
[ 0.97h] [sp] step 250 grad_norm 2.715e-04 prefix_norm 41.6687
[ 1.14h] [sp] step 300 grad_norm 2.842e-04 prefix_norm 46.9475
[ 1.32h] [sp] step 350 grad_norm 1.380e-04 prefix_norm 52.4299
[ 1.50h] [sp] step 400 grad_norm 2.072e-04 prefix_norm 58.1630
[ 1.68h] [sp] step 450 grad_norm 1.345e-04 prefix_norm 64.6393
[ 1.88h] 08/01/2026 19:53:36 - INFO - __main__ - [R

CompletedProcess(args=['cp', '/content/sp_iii_p139_console.log', "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/soft_prompt/experiments/sp_iii_p139/console.log"], returncode=0)

## 11. Checkpoint inventory

In [ ]:
import glob
files = sorted(glob.glob(f"{EXP}/prompt-*.pt"), key=lambda f: int(re.search(r"prompt-(\d+)", f).group(1)))
print(f"{'step':>6s} {'MB':>7s} {'prefix norm':>12s} {'drift from init':>16s}")
init_n = init.norm().item()
for f in files:
    d = torch.load(f, map_location="cpu")
    n = d["prefix"].float().norm().item()
    print(f"{d['step']:>6d} {os.path.getsize(f)/1e6:>7.2f} {n:>12.4f} {100*(n-init_n)/init_n:>15.1f}%")
assert len(files) == STEPS // CKPT_EVERY, f"expected {STEPS//CKPT_EVERY} checkpoints, found {len(files)}"
json.dump({"run": RUN, "P": P, "lr": LR, "steps": STEPS, "seed": SEED,
           "trainable": int(init.numel()), "checkpoints": [int(re.search(r"prompt-(\d+)", f).group(1)) for f in files],
           "train_rows": len(rows)}, open(f"{EXP}/manifest.json", "w"), indent=1)
print(f"\nmanifest written to {EXP}/manifest.json")

  step      MB  prefix norm  drift from init
   500   10.25      70.1466           119.0%
  1000   10.25     107.8046           236.6%
  1500   10.25     129.3080           303.7%
  2000   10.25     149.0196           365.2%
  2500   10.25     164.7436           414.3%
  3000   10.25     180.0249           462.0%

manifest written to /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/soft_prompt/experiments/sp_iii_p139/manifest.json


## 12. Revert the patch

In [ ]:
subprocess.run(["git","-C",REPO,"checkout","--","train.py"], check=True)
s = open(f"{REPO}/train.py", encoding="utf-8").read()
assert "[soft_prompt]" not in s and "[sp] saved" not in s
assert "loss * accelerator.gradient_state.num_steps" in s, "loss normalisation not restored"
print("train.py reverted to HEAD")

train.py reverted to HEAD
